In [1]:
import os
import pandas as pd
import numpy as np

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [2]:
load_folder = "Data"

In [3]:
data = {}

for i in range(1, 9):
    file_path = os.path.join(
        load_folder,
        f"function_{i}.csv"
    )

    data[i] = pd.read_csv(file_path)

    print(
        f"Function {i}: "
        f"{len(data[i])} observations, "
        f"{len(data[i].columns) - 1} dimensions"
    )

Function 1: 13 observations, 2 dimensions
Function 2: 13 observations, 2 dimensions
Function 3: 18 observations, 3 dimensions
Function 4: 33 observations, 4 dimensions
Function 5: 23 observations, 4 dimensions
Function 6: 23 observations, 5 dimensions
Function 7: 33 observations, 6 dimensions
Function 8: 43 observations, 8 dimensions


In [4]:
new_inputs = {
    1: [0.417430, 0.463246],
    2: [0.730707, 0.915532],
    3: [0.433866, 0.716516, 0.000863],
    4: [0.443740, 0.493644, 0.409790, 0.452219],
    5: [0.851064, 0.985689, 0.919598, 0.979330],
    6: [0.033216, 0.058439, 0.124012, 0.011504, 0.042834],
    7: [0.154752, 0.313081, 0.341855, 0.169387, 0.318877, 0.633022],
    8: [0.148540, 0.297539, 0.201860, 0.328776,
        0.636564, 0.500531, 0.197949, 0.479426]
}

new_outputs = {
    1: -0.005912680468954885,
    2: 0.5074182645857468,
    3: -0.10720630327643242,
    4: -1.4386431207074888,
    5: 4857.449601416715,
    6: -2.043879876538988,
    7: 2.536938996958511,
    8: 9.9112493761444
}

In [5]:
previous_bests = {
    i: data[i]["output"].max()
    for i in range(1, 9)
}

In [6]:
for i in range(1, 9):

    feature_columns = [
        col for col in data[i].columns
        if col != "output"
    ]

    new_row = dict(
        zip(feature_columns, new_inputs[i])
    )
    new_row["output"] = new_outputs[i]

    data[i] = pd.concat(
        [data[i], pd.DataFrame([new_row])],
        ignore_index=True
    )

In [7]:
for i in range(1, 9):

    previous_best = previous_bests[i]
    new_output = new_outputs[i]
    updated_best = data[i]["output"].max()

    difference = new_output - previous_best

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print(f"Previous best: {previous_best:.6f}")
    print(f"New output:    {new_output:.6f}")
    print(f"Difference:    {difference:.6f}")
    print(f"Updated best:  {updated_best:.6f}")

    if new_output > previous_best:
        print("Result: NEW BEST")
    else:
        print("Result: Did not improve")

    print()

FUNCTION 1
Previous best: 0.000000
New output:    -0.005913
Difference:    -0.005913
Updated best:  0.000000
Result: Did not improve

FUNCTION 2
Previous best: 0.611205
New output:    0.507418
Difference:    -0.103787
Updated best:  0.611205
Result: Did not improve

FUNCTION 3
Previous best: -0.020167
New output:    -0.107206
Difference:    -0.087039
Updated best:  -0.020167
Result: Did not improve

FUNCTION 4
Previous best: -0.300645
New output:    -1.438643
Difference:    -1.137998
Updated best:  -0.300645
Result: Did not improve

FUNCTION 5
Previous best: 4595.479318
New output:    4857.449601
Difference:    261.970283
Updated best:  4857.449601
Result: NEW BEST

FUNCTION 6
Previous best: -0.458511
New output:    -2.043880
Difference:    -1.585369
Updated best:  -0.458511
Result: Did not improve

FUNCTION 7
Previous best: 2.139551
New output:    2.536939
Difference:    0.397388
Updated best:  2.536939
Result: NEW BEST

FUNCTION 8
Previous best: 9.821986
New output:    9.911249
Diffe

In [8]:
models_week5 = {}

for i in range(1, 9):

    X = data[i].drop(columns="output").values
    y = data[i]["output"].values

    kernel = RBF(
        length_scale=1.0,
        length_scale_bounds=(1e-5, 1e5)
    )

    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    gp.fit(X, y)

    models_week5[i] = gp

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)
    print(f"Fitted kernel: {gp.kernel_}")
    print()

FUNCTION 1
Fitted kernel: RBF(length_scale=1e-05)

FUNCTION 2
Fitted kernel: RBF(length_scale=0.0494)

FUNCTION 3
Fitted kernel: RBF(length_scale=0.173)

FUNCTION 4
Fitted kernel: RBF(length_scale=0.414)

FUNCTION 5
Fitted kernel: RBF(length_scale=0.467)

FUNCTION 6
Fitted kernel: RBF(length_scale=1e-05)

FUNCTION 7
Fitted kernel: RBF(length_scale=0.33)

FUNCTION 8
Fitted kernel: RBF(length_scale=0.979)



In [9]:
rng = np.random.default_rng(42)

gp_diagnostics = {}

for i in range(1, 9):

    n_dimensions = len(data[i].columns) - 1

    # Generate candidate points across the search space
    candidates = rng.random((100000, n_dimensions))

    # GP predictions
    mean_pred, std_pred = models_week5[i].predict(
        candidates,
        return_std=True
    )

    mean_range = mean_pred.max() - mean_pred.min()
    std_range = std_pred.max() - std_pred.min()

    gp_diagnostics[i] = {
        "mean_min": mean_pred.min(),
        "mean_max": mean_pred.max(),
        "mean_range": mean_range,
        "std_min": std_pred.min(),
        "std_max": std_pred.max(),
        "std_range": std_range
    }

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print(f"Predicted mean minimum: {mean_pred.min():.10f}")
    print(f"Predicted mean maximum: {mean_pred.max():.10f}")
    print(f"Predicted mean range:   {mean_range:.10f}")
    print()
    print(f"Uncertainty minimum:    {std_pred.min():.10f}")
    print(f"Uncertainty maximum:    {std_pred.max():.10f}")
    print(f"Uncertainty range:      {std_range:.10f}")
    print()

FUNCTION 1
Predicted mean minimum: -0.0006799102
Predicted mean maximum: -0.0000000377
Predicted mean range:   0.0006798725

Uncertainty minimum:    0.0000181309
Uncertainty maximum:    0.0017215356
Uncertainty range:      0.0017034047

FUNCTION 2
Predicted mean minimum: -0.0822506899
Predicted mean maximum: 0.6168811209
Predicted mean range:   0.6991318109

Uncertainty minimum:    0.0000031216
Uncertainty maximum:    0.2374310817
Uncertainty range:      0.2374279601

FUNCTION 3
Predicted mean minimum: -0.3973810066
Predicted mean maximum: -0.0082630242
Predicted mean range:   0.3891179823

Uncertainty minimum:    0.0000008049
Uncertainty maximum:    0.0779100026
Uncertainty range:      0.0779091977

FUNCTION 4
Predicted mean minimum: -34.4842397129
Predicted mean maximum: -0.0236981328
Predicted mean range:   34.4605415801

Uncertainty minimum:    0.0000826978
Uncertainty maximum:    7.3165011327
Uncertainty range:      7.3164184349

FUNCTION 5
Predicted mean minimum: -1205.5152412033

In [10]:
from scipy.stats import norm

def expected_improvement(mean, std, best_y, xi=0.01):
    improvement = mean - best_y - xi

    with np.errstate(divide="ignore", invalid="ignore"):
        Z = improvement / std
        ei = (
            improvement * norm.cdf(Z)
            + std * norm.pdf(Z)
        )

    ei[std == 0] = 0
    return ei

In [11]:
kappa = 1.5
xi = 0.01
n_candidates = 100000

gp_candidate_results = {}

for i in range(2, 9):

    X_existing = data[i].drop(columns="output").values
    y_existing = data[i]["output"].values

    n_dimensions = X_existing.shape[1]

    # Same candidate set for EI and UCB
    rng = np.random.default_rng(100 + i)

    X_candidates = rng.uniform(
        0,
        1,
        size=(n_candidates, n_dimensions)
    )

    mean, std = models_week5[i].predict(
        X_candidates,
        return_std=True
    )

    best_index = np.argmax(y_existing)
    best_y = y_existing[best_index]
    best_point = X_existing[best_index]

    # -------------------------
    # UCB
    # -------------------------

    ucb = mean + kappa * std
    ucb_index = np.argmax(ucb)
    ucb_point = X_candidates[ucb_index]

    # -------------------------
    # EI
    # -------------------------

    ei = expected_improvement(
        mean,
        std,
        best_y,
        xi=xi
    )

    ei_index = np.argmax(ei)
    ei_point = X_candidates[ei_index]

    # -------------------------
    # Distances
    # -------------------------

    ucb_distance_best = np.linalg.norm(
        ucb_point - best_point
    )

    ei_distance_best = np.linalg.norm(
        ei_point - best_point
    )

    ucb_nearest = np.linalg.norm(
        X_existing - ucb_point,
        axis=1
    ).min()

    ei_nearest = np.linalg.norm(
        X_existing - ei_point,
        axis=1
    ).min()

    distance_between = np.linalg.norm(
        ucb_point - ei_point
    )

    gp_candidate_results[i] = {
        "ucb_point": ucb_point,
        "ucb_mean": mean[ucb_index],
        "ucb_std": std[ucb_index],
        "ucb_score": ucb[ucb_index],
        "ucb_distance_best": ucb_distance_best,
        "ucb_nearest": ucb_nearest,

        "ei_point": ei_point,
        "ei_mean": mean[ei_index],
        "ei_std": std[ei_index],
        "ei_score": ei[ei_index],
        "ei_distance_best": ei_distance_best,
        "ei_nearest": ei_nearest,

        "distance_between": distance_between
    }

    print("=" * 70)
    print(f"FUNCTION {i}")
    print("=" * 70)

    print("UCB (κ = 1.5)")
    print(np.round(ucb_point, 6))
    print(f"Predicted output:               {mean[ucb_index]:.6f}")
    print(f"Uncertainty:                    {std[ucb_index]:.6f}")
    print(f"UCB score:                      {ucb[ucb_index]:.6f}")
    print(f"Distance from current best:     {ucb_distance_best:.4f}")
    print(f"Distance from nearest existing: {ucb_nearest:.4f}")

    print()

    print("EXPECTED IMPROVEMENT")
    print(np.round(ei_point, 6))
    print(f"Predicted output:               {mean[ei_index]:.6f}")
    print(f"Uncertainty:                    {std[ei_index]:.6f}")
    print(f"EI score:                       {ei[ei_index]:.6f}")
    print(f"Distance from current best:     {ei_distance_best:.4f}")
    print(f"Distance from nearest existing: {ei_nearest:.4f}")

    print()

    print(
        f"Distance between UCB and EI candidates: "
        f"{distance_between:.4f}"
    )

    print()

FUNCTION 2
UCB (κ = 1.5)
[0.713179 0.112872]
Predicted output:               0.431873
Uncertainty:                    0.187151
UCB score:                      0.712599
Distance from current best:     0.8138
Distance from nearest existing: 0.0487

EXPECTED IMPROVEMENT
[0.713092 0.138155]
Predicted output:               0.429437
Uncertainty:                    0.188754
EI score:                       0.015254
Distance from current best:     0.7885
Distance from nearest existing: 0.0494

Distance between UCB and EI candidates: 0.0253

FUNCTION 3
UCB (κ = 1.5)
[0.414712 0.608726 0.505958]
Predicted output:               -0.034992
Uncertainty:                    0.055091
UCB score:                      0.047645
Distance from current best:     0.2102
Distance from nearest existing: 0.1832

EXPECTED IMPROVEMENT
[0.384686 0.478261 0.416295]
Predicted output:               -0.013239
Uncertainty:                    0.036050
EI score:                       0.012898
Distance from current best:    

In [12]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.preprocessing import StandardScaler

In [13]:
class SimpleNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x)

In [14]:
nn_models = {}
nn_scalers = {}
nn_training_losses = {}

n_networks = 5
n_epochs = 1000
learning_rate = 0.01

for i in range(1, 9):

    X = data[i].drop(columns="output").values
    y = data[i]["output"].values.reshape(-1, 1)

    # Standardise inputs and outputs
    X_scaler = StandardScaler()
    y_scaler = StandardScaler()

    X_scaled = X_scaler.fit_transform(X)
    y_scaled = y_scaler.fit_transform(y)

    X_tensor = torch.tensor(
        X_scaled,
        dtype=torch.float32
    )

    y_tensor = torch.tensor(
        y_scaled,
        dtype=torch.float32
    )

    function_models = []
    function_losses = []

    for seed in range(n_networks):

        torch.manual_seed(seed)

        model = SimpleNN(
            input_dim=X.shape[1]
        )

        optimiser = torch.optim.Adam(
            model.parameters(),
            lr=learning_rate
        )

        loss_function = nn.MSELoss()

        for epoch in range(n_epochs):

            optimiser.zero_grad()

            predictions = model(X_tensor)

            loss = loss_function(
                predictions,
                y_tensor
            )

            loss.backward()
            optimiser.step()

        function_models.append(model)
        function_losses.append(loss.item())

    nn_models[i] = function_models

    nn_scalers[i] = {
        "X": X_scaler,
        "y": y_scaler
    }

    nn_training_losses[i] = function_losses

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print(
        f"Mean final training loss: "
        f"{np.mean(function_losses):.6f}"
    )

    print(
        f"Minimum final training loss: "
        f"{np.min(function_losses):.6f}"
    )

    print(
        f"Maximum final training loss: "
        f"{np.max(function_losses):.6f}"
    )

    print()

FUNCTION 1
Mean final training loss: 0.001044
Minimum final training loss: 0.000000
Maximum final training loss: 0.005221

FUNCTION 2
Mean final training loss: 0.002267
Minimum final training loss: 0.000000
Maximum final training loss: 0.010432

FUNCTION 3
Mean final training loss: 0.000001
Minimum final training loss: 0.000000
Maximum final training loss: 0.000003

FUNCTION 4
Mean final training loss: 0.000000
Minimum final training loss: 0.000000
Maximum final training loss: 0.000001

FUNCTION 5
Mean final training loss: 0.000004
Minimum final training loss: 0.000000
Maximum final training loss: 0.000016

FUNCTION 6
Mean final training loss: 0.000013
Minimum final training loss: 0.000000
Maximum final training loss: 0.000034

FUNCTION 7
Mean final training loss: 0.000000
Minimum final training loss: 0.000000
Maximum final training loss: 0.000002

FUNCTION 8
Mean final training loss: 0.000000
Minimum final training loss: 0.000000
Maximum final training loss: 0.000000



In [15]:
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_absolute_error

nn_cv_results = {}

for i in range(1, 9):

    X = data[i].drop(columns="output").values
    y = data[i]["output"].values

    loo = LeaveOneOut()

    actual_values = []
    nn_predictions = []
    baseline_predictions = []

    for fold, (train_index, test_index) in enumerate(loo.split(X)):

        X_train = X[train_index]
        X_test = X[test_index]

        y_train = y[train_index]
        y_test = y[test_index]

        # Scale using training data only
        X_scaler = StandardScaler()
        y_scaler = StandardScaler()

        X_train_scaled = X_scaler.fit_transform(X_train)
        X_test_scaled = X_scaler.transform(X_test)

        y_train_scaled = y_scaler.fit_transform(
            y_train.reshape(-1, 1)
        )

        X_train_tensor = torch.tensor(
            X_train_scaled,
            dtype=torch.float32
        )

        y_train_tensor = torch.tensor(
            y_train_scaled,
            dtype=torch.float32
        )

        X_test_tensor = torch.tensor(
            X_test_scaled,
            dtype=torch.float32
        )

        # Reproducible network for this fold
        torch.manual_seed(42 + fold)

        model = SimpleNN(
            input_dim=X.shape[1]
        )

        optimiser = torch.optim.Adam(
            model.parameters(),
            lr=learning_rate
        )

        loss_function = nn.MSELoss()

        for epoch in range(n_epochs):

            optimiser.zero_grad()

            predictions = model(X_train_tensor)

            loss = loss_function(
                predictions,
                y_train_tensor
            )

            loss.backward()
            optimiser.step()

        # Predict held-out observation
        model.eval()

        with torch.no_grad():

            prediction_scaled = (
                model(X_test_tensor)
                .numpy()
                .reshape(-1, 1)
            )

        prediction = y_scaler.inverse_transform(
            prediction_scaled
        )[0, 0]

        actual_values.append(y_test[0])
        nn_predictions.append(prediction)

        # Simple mean baseline
        baseline_predictions.append(
            np.mean(y_train)
        )

    nn_mae = mean_absolute_error(
        actual_values,
        nn_predictions
    )

    baseline_mae = mean_absolute_error(
        actual_values,
        baseline_predictions
    )

    nn_cv_results[i] = {
        "nn_mae": nn_mae,
        "baseline_mae": baseline_mae
    }

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)
    print(f"NN LOOCV MAE:       {nn_mae:.6f}")
    print(f"Mean baseline MAE:  {baseline_mae:.6f}")
    print(
        "NN beats baseline: ",
        nn_mae < baseline_mae
    )
    print()

FUNCTION 1
NN LOOCV MAE:       0.002595
Mean baseline MAE:  0.001255
NN beats baseline:  False

FUNCTION 2
NN LOOCV MAE:       0.188740
Mean baseline MAE:  0.225146
NN beats baseline:  True

FUNCTION 3
NN LOOCV MAE:       0.066222
Mean baseline MAE:  0.046560
NN beats baseline:  False

FUNCTION 4
NN LOOCV MAE:       1.998674
Mean baseline MAE:  6.651296
NN beats baseline:  True

FUNCTION 5
NN LOOCV MAE:       192.581456
Mean baseline MAE:  1041.351340
NN beats baseline:  True

FUNCTION 6
NN LOOCV MAE:       0.327649
Mean baseline MAE:  0.453136
NN beats baseline:  True

FUNCTION 7
NN LOOCV MAE:       0.315686
Mean baseline MAE:  0.474058
NN beats baseline:  True

FUNCTION 8
NN LOOCV MAE:       0.328345
Mean baseline MAE:  0.900332
NN beats baseline:  True



In [16]:
nn_candidate_results = {}

n_candidates = 100000

for i in range(1, 9):

    X_existing = data[i].drop(columns="output").values
    n_dimensions = X_existing.shape[1]

    # Reproducible candidate set
    rng = np.random.default_rng(200 + i)

    X_candidates = rng.uniform(
        0,
        1,
        size=(n_candidates, n_dimensions)
    )

    # Scale candidates using the scaler fitted in Step 4A
    X_scaler = nn_scalers[i]["X"]
    y_scaler = nn_scalers[i]["y"]

    X_candidates_scaled = X_scaler.transform(X_candidates)

    X_tensor = torch.tensor(
        X_candidates_scaled,
        dtype=torch.float32
    )

    network_predictions = []

    # Get predictions from each of the 5 networks
    for model in nn_models[i]:

        model.eval()

        with torch.no_grad():

            pred_scaled = (
                model(X_tensor)
                .numpy()
                .reshape(-1, 1)
            )

        # Convert back to original output scale
        pred_original = y_scaler.inverse_transform(
            pred_scaled
        ).ravel()

        network_predictions.append(pred_original)

    network_predictions = np.array(network_predictions)

    # Mean prediction and disagreement across networks
    mean_prediction = network_predictions.mean(axis=0)
    std_prediction = network_predictions.std(axis=0)

    # Candidate with highest average predicted output
    best_index = np.argmax(mean_prediction)

    selected_point = X_candidates[best_index]
    selected_mean = mean_prediction[best_index]
    selected_std = std_prediction[best_index]

    # Distances for context
    distances_existing = np.linalg.norm(
        X_existing - selected_point,
        axis=1
    )

    nearest_distance = distances_existing.min()

    current_best_index = data[i]["output"].idxmax()

    current_best_point = (
        data[i]
        .drop(columns="output")
        .loc[current_best_index]
        .values
        .astype(float)
    )

    distance_best = np.linalg.norm(
        selected_point - current_best_point
    )

    nn_candidate_results[i] = {
        "point": selected_point,
        "predicted_mean": selected_mean,
        "prediction_std": selected_std,
        "nearest_distance": nearest_distance,
        "distance_best": distance_best
    }

    print("=" * 70)
    print(f"FUNCTION {i}")
    print("=" * 70)

    print("NN selected point:")
    print(np.round(selected_point, 6))

    print(f"NN predicted output:             {selected_mean:.6f}")
    print(f"NN model disagreement:          {selected_std:.6f}")
    print(f"Distance from current best:     {distance_best:.4f}")
    print(f"Distance from nearest existing: {nearest_distance:.4f}")

    print()

FUNCTION 1
NN selected point:
[9.78948e-01 6.56000e-04]
NN predicted output:             0.004556
NN model disagreement:          0.001415
Distance from current best:     0.7732
Distance from nearest existing: 0.2982

FUNCTION 2
NN selected point:
[9.98493e-01 5.18000e-04]
NN predicted output:             1.116515
NN model disagreement:          0.831745
Distance from current best:     0.9722
Distance from nearest existing: 0.3549

FUNCTION 3
NN selected point:
[0.297078 0.002134 0.846217]
NN predicted output:             0.000985
NN model disagreement:          0.027674
Distance from current best:     0.5262
Distance from nearest existing: 0.3477

FUNCTION 4
NN selected point:
[0.394245 0.436007 0.403516 0.426489]
NN predicted output:             -0.570388
NN model disagreement:          0.191094
Distance from current best:     0.0578
Distance from nearest existing: 0.0578

FUNCTION 5
NN selected point:
[0.937033 0.990945 0.91798  0.998735]
NN predicted output:             5001.947266

In [17]:
i = 1
kappa = 1.5
xi = 0.01
n_candidates = 100000

X_existing = data[i].drop(columns="output").values
y_existing = data[i]["output"].values

n_dimensions = X_existing.shape[1]

rng = np.random.default_rng(100 + i)

X_candidates = rng.uniform(
    0,
    1,
    size=(n_candidates, n_dimensions)
)

mean, std = models_week5[i].predict(
    X_candidates,
    return_std=True
)

best_index = np.argmax(y_existing)
best_y = y_existing[best_index]
best_point = X_existing[best_index]

# UCB
ucb = mean + kappa * std
ucb_index = np.argmax(ucb)
ucb_point = X_candidates[ucb_index]

# EI
ei = expected_improvement(
    mean,
    std,
    best_y,
    xi=xi
)

ei_index = np.argmax(ei)
ei_point = X_candidates[ei_index]

# Distances
ucb_distance_best = np.linalg.norm(
    ucb_point - best_point
)

ei_distance_best = np.linalg.norm(
    ei_point - best_point
)

ucb_nearest = np.linalg.norm(
    X_existing - ucb_point,
    axis=1
).min()

ei_nearest = np.linalg.norm(
    X_existing - ei_point,
    axis=1
).min()

distance_between = np.linalg.norm(
    ucb_point - ei_point
)

gp_candidate_results[i] = {
    "ucb_point": ucb_point,
    "ucb_mean": mean[ucb_index],
    "ucb_std": std[ucb_index],
    "ucb_score": ucb[ucb_index],
    "ucb_distance_best": ucb_distance_best,
    "ucb_nearest": ucb_nearest,

    "ei_point": ei_point,
    "ei_mean": mean[ei_index],
    "ei_std": std[ei_index],
    "ei_score": ei[ei_index],
    "ei_distance_best": ei_distance_best,
    "ei_nearest": ei_nearest,

    "distance_between": distance_between
}

print("=" * 70)
print("FUNCTION 1")
print("=" * 70)

print("UCB (κ = 1.5)")
print(np.round(ucb_point, 6))
print(f"Predicted output:               {mean[ucb_index]:.6f}")
print(f"Distance from current best:     {ucb_distance_best:.4f}")
print(f"Distance from nearest existing: {ucb_nearest:.4f}")

print()

print("EXPECTED IMPROVEMENT")
print(np.round(ei_point, 6))
print(f"Predicted output:               {mean[ei_index]:.6f}")
print(f"Distance from current best:     {ei_distance_best:.4f}")
print(f"Distance from nearest existing: {ei_nearest:.4f}")

print()

print(
    f"Distance between UCB and EI candidates: "
    f"{distance_between:.4f}"
)

FUNCTION 1
UCB (κ = 1.5)
[0.943533 0.359421]
Predicted output:               -0.000680
Distance from current best:     0.4298
Distance from nearest existing: 0.1400

EXPECTED IMPROVEMENT
[0.943533 0.359421]
Predicted output:               -0.000680
Distance from current best:     0.4298
Distance from nearest existing: 0.1400

Distance between UCB and EI candidates: 0.0000


In [18]:
model_comparison = {}

for i in range(1, 9):

    nn_point = nn_candidate_results[i]["point"]
    ucb_point = gp_candidate_results[i]["ucb_point"]
    ei_point = gp_candidate_results[i]["ei_point"]

    nn_to_ucb = np.linalg.norm(
        nn_point - ucb_point
    )

    nn_to_ei = np.linalg.norm(
        nn_point - ei_point
    )

    ucb_to_ei = np.linalg.norm(
        ucb_point - ei_point
    )

    model_comparison[i] = {
        "nn_to_ucb": nn_to_ucb,
        "nn_to_ei": nn_to_ei,
        "ucb_to_ei": ucb_to_ei
    }

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print(f"NN → UCB distance: {nn_to_ucb:.4f}")
    print(f"NN → EI distance:  {nn_to_ei:.4f}")
    print(f"UCB → EI distance: {ucb_to_ei:.4f}")

    print()

FUNCTION 1
NN → UCB distance: 0.3605
NN → EI distance:  0.3605
UCB → EI distance: 0.0000

FUNCTION 2
NN → UCB distance: 0.3066
NN → EI distance:  0.3169
UCB → EI distance: 0.0253

FUNCTION 3
NN → UCB distance: 0.7054
NN → EI distance:  0.6475
UCB → EI distance: 0.1611

FUNCTION 4
NN → UCB distance: 0.1381
NN → EI distance:  0.1198
UCB → EI distance: 0.0193

FUNCTION 5
NN → UCB distance: 0.0966
NN → EI distance:  0.0718
UCB → EI distance: 0.0444

FUNCTION 6
NN → UCB distance: 1.3150
NN → EI distance:  1.3150
UCB → EI distance: 0.0000

FUNCTION 7
NN → UCB distance: 0.2738
NN → EI distance:  0.2738
UCB → EI distance: 0.0000

FUNCTION 8
NN → UCB distance: 0.7429
NN → EI distance:  0.7940
UCB → EI distance: 0.7320



In [19]:
for i in range(1, 9):

    X_existing = data[i].drop(columns="output").values
    y_existing = data[i]["output"].values

    candidates = {
        "UCB": gp_candidate_results[i]["ucb_point"],
        "EI": gp_candidate_results[i]["ei_point"],
        "NN": nn_candidate_results[i]["point"]
    }

    print("=" * 70)
    print(f"FUNCTION {i}")
    print("=" * 70)

    for method, candidate in candidates.items():

        distances = np.linalg.norm(
            X_existing - candidate,
            axis=1
        )

        nearest_indices = np.argsort(distances)[:3]

        print(f"\n{method} candidate:")
        print(np.round(candidate, 6))

        print("Nearest observations:")

        for idx in nearest_indices:
            print(
                f"  Distance = {distances[idx]:.4f} | "
                f"Output = {y_existing[idx]:.6f}"
            )

    print()

FUNCTION 1

UCB candidate:
[0.943533 0.359421]
Nearest observations:
  Distance = 0.1400 | Output = 0.000000
  Distance = 0.2307 | Output = 0.000000
  Distance = 0.4298 | Output = 0.000000

EI candidate:
[0.943533 0.359421]
Nearest observations:
  Distance = 0.1400 | Output = 0.000000
  Distance = 0.2307 | Output = 0.000000
  Distance = 0.4298 | Output = 0.000000

NN candidate:
[9.78948e-01 6.56000e-04]
Nearest observations:
  Distance = 0.2982 | Output = 0.000000
  Distance = 0.5872 | Output = -0.000000
  Distance = 0.5893 | Output = 0.000000

FUNCTION 2

UCB candidate:
[0.713179 0.112872]
Nearest observations:
  Distance = 0.0487 | Output = 0.538996
  Distance = 0.3136 | Output = 0.214965
  Distance = 0.3808 | Output = 0.038749

EI candidate:
[0.713092 0.138155]
Nearest observations:
  Distance = 0.0494 | Output = 0.538996
  Distance = 0.3000 | Output = 0.214965
  Distance = 0.3820 | Output = -0.013858

NN candidate:
[9.98493e-01 5.18000e-04]
Nearest observations:
  Distance = 0.3549

In [20]:
final_selections = {
    1: {
        "point": gp_candidate_results[1]["ei_point"],
        "method": "EI / UCB agreement"
    },
    2: {
        "point": gp_candidate_results[2]["ei_point"],
        "method": "Expected Improvement"
    },
    3: {
        "point": gp_candidate_results[3]["ei_point"],
        "method": "Expected Improvement"
    },
    4: {
        "point": gp_candidate_results[4]["ei_point"],
        "method": "Expected Improvement"
    },
    5: {
        "point": gp_candidate_results[5]["ei_point"],
        "method": "Expected Improvement"
    },
    6: {
        "point": nn_candidate_results[6]["point"],
        "method": "Neural Network"
    },
    7: {
        "point": gp_candidate_results[7]["ei_point"],
        "method": "EI / UCB agreement"
    },
    8: {
        "point": nn_candidate_results[8]["point"],
        "method": "Neural Network"
    }
}

print("=" * 70)
print("FINAL WEEK 5 VALIDATION")
print("=" * 70)

all_valid = True

for i in range(1, 9):

    point = np.asarray(
        final_selections[i]["point"],
        dtype=float
    )

    method = final_selections[i]["method"]

    X_existing = (
        data[i]
        .drop(columns="output")
        .values
        .astype(float)
    )

    # Check bounds
    in_bounds = np.all(
        (point >= 0) & (point <= 1)
    )

    # Check for exact/near duplicate
    distances = np.linalg.norm(
        X_existing - point,
        axis=1
    )

    nearest_distance = distances.min()

    duplicate = np.any(
        np.all(
            np.isclose(
                X_existing,
                point,
                atol=1e-8
            ),
            axis=1
        )
    )

    valid = in_bounds and not duplicate

    if not valid:
        all_valid = False

    print(f"\nFUNCTION {i}")
    print(f"Selected point: {np.round(point, 6)}")
    print(f"Selection method: {method}")
    print(f"In bounds: {in_bounds}")
    print(f"Duplicate existing point: {duplicate}")
    print(
        f"Distance to nearest observation: "
        f"{nearest_distance:.6f}"
    )

print("\n" + "=" * 70)
print(f"ALL EIGHT POINTS VALID: {all_valid}")
print("=" * 70)

FINAL WEEK 5 VALIDATION

FUNCTION 1
Selected point: [0.943533 0.359421]
Selection method: EI / UCB agreement
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.140043

FUNCTION 2
Selected point: [0.713092 0.138155]
Selection method: Expected Improvement
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.049374

FUNCTION 3
Selected point: [0.384686 0.478261 0.416295]
Selection method: Expected Improvement
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.132463

FUNCTION 4
Selected point: [0.462451 0.342242 0.380323 0.445636]
Selection method: Expected Improvement
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.114172

FUNCTION 5
Selected point: [0.938942 0.989452 0.984552 0.971823]
Selection method: Expected Improvement
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.109600

FUNCTION 6
Selected point: [0.248909 0.369411 0

In [21]:
print("WEEK 5 SUBMISSION POINTS")
print("=" * 70)

for i in range(1, 9):

    point = np.asarray(
        final_selections[i]["point"],
        dtype=float
    )

    formatted = "-".join(
        f"{x:.6f}" for x in point
    )

    print(f"Function {i}: {formatted}")

WEEK 5 SUBMISSION POINTS
Function 1: 0.943533-0.359421
Function 2: 0.713092-0.138155
Function 3: 0.384686-0.478261-0.416295
Function 4: 0.462451-0.342242-0.380323-0.445636
Function 5: 0.938942-0.989452-0.984552-0.971823
Function 6: 0.248909-0.369411-0.484746-0.674732-0.011701
Function 7: 0.132842-0.294043-0.285122-0.050957-0.281982-0.619768
Function 8: 0.034135-0.275291-0.032179-0.508682-0.760754-0.617950-0.034127-0.349420
